# **Project Name**    - Voyage Analytics: Travel Price Prediction and Recommendation

##### **Project Type**    - EDA/Regression/Classification/Recommendation/MLOps
##### **Contribution**    - Individual
##### **Team Member 1 -** Harsh

# **Project Summary -**

This project analyzes integrated travel datasets containing users, flights, and hotel bookings to understand customer travel behavior and build machine learning solutions for travel decision support. The users table provides demographic and company information, the flights table captures route, agency, class, distance, duration, date, and ticket price, and the hotels table records hotel, destination, length of stay, per-day price, and total booking value. Together, these datasets make it possible to study demand by city, route economics, customer stay patterns, and the relationship between booking attributes and cost.

The exploratory phase begins by loading all three datasets, checking shape, data types, missing values, duplicates, and summary statistics. Date columns are converted into calendar features so travel activity can be compared by month and day of week. The analysis joins flights and hotels through travelCode and userCode, creating a trip-level view that supports insights across transportation and accommodation. Visualizations examine flight price distribution, route popularity, agency pricing, flight class impact, distance-price behavior, hotel demand by destination, stay duration, hotel revenue, user demographics, seasonality, and correlations among numeric features.

The machine learning phase focuses on the primary objective: predicting flight price from operational attributes such as origin, destination, flight class, agency, flight time, distance, and booking date. Several regression models are compared using MAE, RMSE, and R2. A preprocessing pipeline handles numeric imputation and scaling, categorical imputation and one-hot encoding, and model fitting in a reproducible way. Random Forest is selected as the final model because it captures nonlinear route and agency interactions and produces the strongest validation performance on this structured dataset. MLflow is used for experiment tracking and joblib is used to serialize the final model for deployment.

Additional capstone objectives are also addressed. A gender classification model is built from the users table as a deployment exercise, while carefully excluding direct identifiers such as user code and name. Because only age and company remain as predictive fields, this model is expected to have limited practical accuracy and should not be treated as a high-confidence demographic inference system. A hotel recommendation component ranks hotel options using historical popularity, destination, price, and user history. Finally, the project includes deployment-ready assets: Flask REST API, Dockerfile, Kubernetes manifests, Airflow DAG, Jenkins pipeline, MLflow tracking, and a Streamlit app for hotel recommendations and visual exploration.

# **GitHub Link -**

GitHub Link: This notebook is part of the local Voyage Analytics capstone workspace. After publishing, replace this line with the final repository URL.

# **Problem Statement**


Travel companies need reliable ways to estimate flight prices, understand booking behavior, and recommend relevant hotels using historical travel data. Manual analysis does not scale well across many routes, agencies, users, destinations, and seasonal patterns. The goal is to use the users, flights, and hotels datasets to discover actionable business insights and build deployable machine learning components for flight price prediction, user gender classification, and hotel recommendation.

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required. 
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits. 
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule. 

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

pd.set_option('display.max_columns', 100)
plt.style.use('seaborn-v0_8-whitegrid')

DATA_DIR = Path('.')
FLIGHTS_PATH = DATA_DIR / 'flights.csv'
HOTELS_PATH = DATA_DIR / 'hotels.csv'
USERS_PATH = DATA_DIR / 'users.csv'
MODELS_DIR = DATA_DIR / 'models'
MODELS_DIR.mkdir(exist_ok=True)

from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


### Dataset Loading

In [ ]:
# Load Dataset
flights = pd.read_csv(FLIGHTS_PATH)
hotels = pd.read_csv(HOTELS_PATH)
users = pd.read_csv(USERS_PATH)

datasets = {'flights': flights, 'hotels': hotels, 'users': users}
print({name: df.shape for name, df in datasets.items()})


### Dataset First View

In [ ]:
# Dataset First Look
for name, df in datasets.items():
    print(f'\n{name.upper()}')
    display(df.head())


### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
shape_summary = pd.DataFrame(
    [{'dataset': name, 'rows': df.shape[0], 'columns': df.shape[1]} for name, df in datasets.items()]
)
display(shape_summary)


### Dataset Information

In [ ]:
# Dataset Info
for name, df in datasets.items():
    print(f'\n{name.upper()} INFO')
    df.info()


#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
duplicate_summary = pd.DataFrame(
    [{'dataset': name, 'duplicate_rows': int(df.duplicated().sum())} for name, df in datasets.items()]
)
display(duplicate_summary)


#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
missing_summary = pd.concat(
    {name: df.isna().sum() for name, df in datasets.items()},
    axis=1
).fillna(0).astype(int)
display(missing_summary)


In [ ]:
# Visualizing the missing values
missing_pct = pd.DataFrame(
    [{'dataset': name, 'missing_percent': df.isna().mean().mean() * 100} for name, df in datasets.items()]
)
ax = missing_pct.plot(kind='bar', x='dataset', y='missing_percent', legend=False, figsize=(7, 4), color='#4C78A8')
ax.set_ylabel('Average missing values (%)')
ax.set_title('Missing Value Rate by Dataset')
plt.xticks(rotation=0)
plt.show()


The project has three related datasets. Flights has the largest number of rows and is the primary modeling table for regression. Hotels and users enrich business understanding and support recommendation/classification tasks.

The datasets are suitable for supervised learning after preprocessing. Flight price is a continuous target, while route, agency, class, distance, time, and date-derived features provide strong predictive signal. Hotel and user data support secondary recommendation and classification objectives.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
for name, df in datasets.items():
    print(f'{name}: {list(df.columns)}')


In [ ]:
# Dataset Describe
for name, df in datasets.items():
    print(f'\n{name.upper()} NUMERIC SUMMARY')
    display(df.describe(include='all').T)


Flights include travelCode, userCode, origin, destination, flight class, price, duration, distance, agency, and date. Hotels include travelCode, userCode, hotel name, place, stay days, daily price, total price, and booking date. Users include user code, company, name, gender, and age.

The variables include structured categorical features, numeric travel measurements, date fields, and identifiers. The ML workflow removes direct identifiers, converts dates into month/day/year features, imputes missing values, scales numeric fields, and one-hot encodes nominal categories.

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
unique_counts = []
for name, df in datasets.items():
    for col in df.columns:
        unique_counts.append({'dataset': name, 'column': col, 'unique_values': df[col].nunique(dropna=True)})
unique_counts = pd.DataFrame(unique_counts)
display(unique_counts)


## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.
flights_clean = flights.copy()
hotels_clean = hotels.copy()
users_clean = users.copy()

for df in [flights_clean, hotels_clean]:
    df['date'] = pd.to_datetime(df['date'], format='%m/%d/%Y', errors='coerce')
    df['month'] = df['date'].dt.month
    df['day_of_week'] = df['date'].dt.day_name()
    df['year'] = df['date'].dt.year

flights_clean['route'] = flights_clean['from'] + ' -> ' + flights_clean['to']
flights_clean['price_per_km'] = flights_clean['price'] / flights_clean['distance']
hotels_clean['computed_total'] = hotels_clean['days'] * hotels_clean['price']
hotels_clean['total_difference'] = hotels_clean['total'] - hotels_clean['computed_total']

trip_data = flights_clean.merge(
    hotels_clean,
    on=['travelCode', 'userCode'],
    how='left',
    suffixes=('_flight', '_hotel')
).merge(
    users_clean,
    left_on='userCode',
    right_on='code',
    how='left'
)

print('Prepared flights:', flights_clean.shape)
print('Prepared hotels:', hotels_clean.shape)
print('Prepared trip-level data:', trip_data.shape)
display(trip_data.head())


I converted date fields to datetime, extracted month, year, and day-of-week features, created route and price-per-kilometer fields, validated hotel total values, and joined flights, hotels, and users into a trip-level analytical table. The main insights are that route, distance, class, and agency strongly explain flight price, while hotel demand is concentrated in a limited set of destinations.

I converted date fields to datetime, extracted month, year, and day-of-week features, created route and price-per-kilometer fields, validated hotel total values, and joined flights, hotels, and users into a trip-level analytical table. The main insights are that route, distance, class, and agency strongly explain flight price, while hotel demand is concentrated in a limited set of destinations.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart - 1 visualization code
ax = flights_clean['price'].plot(kind='hist', bins=30, figsize=(8, 4), color='#4C78A8', edgecolor='white')
ax.set_title('Distribution of Flight Prices')
ax.set_xlabel('Flight price')
ax.set_ylabel('Number of flights')
plt.show()

##### 1. Why did you pick the specific chart?

A histogram is suitable for seeing the spread, skew, and concentration of flight prices.

##### 2. What is/are the insight(s) found from the chart?

Flight prices are not uniformly distributed; most observations cluster around common route and class price bands.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Understanding common price bands helps pricing, forecasting, and anomaly monitoring.

#### Chart - 2

In [ ]:
# Chart - 2 visualization code
top_routes = flights_clean['route'].value_counts().head(10).sort_values()
ax = top_routes.plot(kind='barh', figsize=(9, 5), color='#59A14F')
ax.set_title('Top 10 Flight Routes by Booking Count')
ax.set_xlabel('Number of flights')
ax.set_ylabel('Route')
plt.show()

##### 1. Why did you pick the specific chart?

A horizontal bar chart makes route names readable while ranking demand clearly.

##### 2. What is/are the insight(s) found from the chart?

A small group of city pairs contributes a large share of flight activity.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. High-volume routes deserve capacity planning, fare monitoring, and targeted offers.

#### Chart - 3

In [ ]:
# Chart - 3 visualization code
agency_price = flights_clean.groupby('agency')['price'].mean().sort_values(ascending=False)
ax = agency_price.plot(kind='bar', figsize=(8, 4), color='#F28E2B')
ax.set_title('Average Flight Price by Agency')
ax.set_xlabel('Agency')
ax.set_ylabel('Average price')
plt.xticks(rotation=20, ha='right')
plt.show()

##### 1. Why did you pick the specific chart?

A bar chart compares average price across agencies directly.

##### 2. What is/are the insight(s) found from the chart?

Agencies show different average fare levels, suggesting different pricing strategies or route mixes.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Agency-level pricing can inform partnership negotiations and customer-facing recommendations.

#### Chart - 4

In [ ]:
# Chart - 4 visualization code
class_price = flights_clean.groupby('flightType')['price'].mean().sort_values(ascending=False)
ax = class_price.plot(kind='bar', figsize=(7, 4), color='#E15759')
ax.set_title('Average Flight Price by Flight Class')
ax.set_xlabel('Flight class')
ax.set_ylabel('Average price')
plt.xticks(rotation=20, ha='right')
plt.show()

##### 1. Why did you pick the specific chart?

A bar chart is best for comparing categories such as flight classes.

##### 2. What is/are the insight(s) found from the chart?

Premium classes carry higher average prices than economy-style classes.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Class is a key predictive feature and a strong lever for revenue segmentation.

#### Chart - 5

In [ ]:
# Chart - 5 visualization code
fig, ax = plt.subplots(figsize=(8, 5))
for flight_type, data in flights_clean.groupby('flightType'):
    ax.scatter(data['distance'], data['price'], alpha=0.45, s=18, label=flight_type)
ax.set_title('Flight Price vs Distance')
ax.set_xlabel('Distance')
ax.set_ylabel('Price')
ax.legend(title='Flight class')
plt.show()

##### 1. Why did you pick the specific chart?

A scatter plot reveals whether price changes with flight distance.

##### 2. What is/are the insight(s) found from the chart?

Distance and price move together, but route, class, and agency create visible price bands.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Distance is useful, but business rules should also include categorical context.

#### Chart - 6

In [ ]:
# Chart - 6 visualization code
monthly_flights = flights_clean.groupby('month')['travelCode'].count().sort_index()
ax = monthly_flights.plot(kind='line', marker='o', figsize=(8, 4), color='#76B7B2')
ax.set_title('Monthly Flight Booking Volume')
ax.set_xlabel('Month')
ax.set_ylabel('Number of flights')
ax.set_xticks(monthly_flights.index)
plt.show()

##### 1. Why did you pick the specific chart?

A line chart shows time-based booking patterns by month.

##### 2. What is/are the insight(s) found from the chart?

Monthly demand varies, indicating potential seasonality in travel behavior.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Seasonality supports staffing, inventory, and campaign timing decisions.

#### Chart - 7

In [ ]:
# Chart - 7 visualization code
top_places = hotels_clean['place'].value_counts().head(10).sort_values()
ax = top_places.plot(kind='barh', figsize=(9, 5), color='#EDC948')
ax.set_title('Top Hotel Destinations by Booking Count')
ax.set_xlabel('Number of hotel bookings')
ax.set_ylabel('Destination')
plt.show()

##### 1. Why did you pick the specific chart?

A ranked bar chart highlights the most requested hotel destinations.

##### 2. What is/are the insight(s) found from the chart?

Hotel demand is concentrated in a few destinations such as major Brazilian cities.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Destination demand helps hotel sourcing and recommendation coverage.

#### Chart - 8

In [ ]:
# Chart - 8 visualization code
stay_counts = hotels_clean['days'].value_counts().sort_index()
ax = stay_counts.plot(kind='bar', figsize=(8, 4), color='#B07AA1')
ax.set_title('Hotel Stay Duration Distribution')
ax.set_xlabel('Stay duration in days')
ax.set_ylabel('Number of bookings')
plt.xticks(rotation=0)
plt.show()

##### 1. Why did you pick the specific chart?

A count chart is appropriate because stay duration is discrete.

##### 2. What is/are the insight(s) found from the chart?

Most hotel bookings are short stays, commonly clustered around a few day counts.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Stay-duration patterns help package design and inventory forecasting.

#### Chart - 9

In [ ]:
# Chart - 9 visualization code
hotel_revenue = hotels_clean.groupby('name')['total'].sum().sort_values(ascending=False).head(10).sort_values()
ax = hotel_revenue.plot(kind='barh', figsize=(9, 5), color='#FF9DA7')
ax.set_title('Top 10 Hotels by Total Revenue')
ax.set_xlabel('Total revenue')
ax.set_ylabel('Hotel')
plt.show()

##### 1. Why did you pick the specific chart?

A revenue ranking shows business value, not just booking volume.

##### 2. What is/are the insight(s) found from the chart?

Some hotels generate much higher total revenue because of price, volume, or both.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Revenue leaders should be prioritized in recommendation and partnership strategies.

#### Chart - 10

In [ ]:
# Chart - 10 visualization code
ax = users_clean['age'].plot(kind='hist', bins=20, figsize=(8, 4), color='#9C755F', edgecolor='white')
ax.set_title('User Age Distribution')
ax.set_xlabel('Age')
ax.set_ylabel('Number of users')
plt.show()

##### 1. Why did you pick the specific chart?

A histogram quickly shows demographic spread.

##### 2. What is/are the insight(s) found from the chart?

Users cover a broad adult age range, with concentration around working-age travelers.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Age can support segmentation, though it should be used carefully and ethically.

#### Chart - 11

In [ ]:
# Chart - 11 visualization code
gender_counts = users_clean['gender'].value_counts()
ax = gender_counts.plot(kind='bar', figsize=(7, 4), color=['#4C78A8', '#F28E2B'])
ax.set_title('User Count by Gender')
ax.set_xlabel('Gender')
ax.set_ylabel('Number of users')
plt.xticks(rotation=0)
plt.show()

##### 1. Why did you pick the specific chart?

A bar chart clearly compares gender counts.

##### 2. What is/are the insight(s) found from the chart?

The user table contains both male and female users with no severe class dominance.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Limited positive impact. Gender should not drive unfair personalization, but balance matters for model evaluation.

#### Chart - 12

In [ ]:
# Chart - 12 visualization code
top_companies = users_clean['company'].value_counts().head(10).sort_values()
ax = top_companies.plot(kind='barh', figsize=(9, 5), color='#BAB0AC')
ax.set_title('Top 10 Companies by User Count')
ax.set_xlabel('Number of users')
ax.set_ylabel('Company')
plt.show()

##### 1. Why did you pick the specific chart?

A ranked bar chart shows which companies contribute most users.

##### 2. What is/are the insight(s) found from the chart?

Several companies dominate the user base, which may affect travel patterns and pricing exposure.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Company-level patterns can inform B2B account management and corporate travel offers.

#### Chart - 13

In [ ]:
# Chart - 13 visualization code
trip_cost = trip_data.copy()
trip_cost['trip_total'] = trip_cost['price_flight'] + trip_cost['total'].fillna(0)
monthly_trip_cost = trip_cost.groupby('month_flight')['trip_total'].mean().sort_index()
ax = monthly_trip_cost.plot(kind='line', marker='o', figsize=(8, 4), color='#2F4B7C')
ax.set_title('Average Total Trip Cost by Flight Month')
ax.set_xlabel('Month')
ax.set_ylabel('Average trip cost')
ax.set_xticks(monthly_trip_cost.index)
plt.show()

##### 1. Why did you pick the specific chart?

A monthly line chart connects flight timing with total trip cost.

##### 2. What is/are the insight(s) found from the chart?

Average trip cost changes by month, combining flight and hotel effects.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. This supports seasonal budget guidance and trip package pricing.

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code
numeric_cols = ['price_flight', 'time', 'distance', 'price_per_km', 'days', 'price_hotel', 'total', 'age']
corr_data = trip_data[[col for col in numeric_cols if col in trip_data.columns]].corr()
fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(corr_data, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_data.columns)))
ax.set_yticks(range(len(corr_data.columns)))
ax.set_xticklabels(corr_data.columns, rotation=45, ha='right')
ax.set_yticklabels(corr_data.columns)
for i in range(len(corr_data.columns)):
    for j in range(len(corr_data.columns)):
        ax.text(j, i, f'{corr_data.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('Correlation Heatmap of Numeric Travel Features')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

A heatmap summarizes many numeric relationships at once.

##### 2. What is/are the insight(s) found from the chart?

Flight distance, time, and price are strongly related; hotel total is closely tied to days and daily price.

#### Chart - 15 - Pair Plot 

In [ ]:
# Pair Plot visualization code
pair_cols = ['price', 'distance', 'time', 'price_per_km']
pd.plotting.scatter_matrix(
    flights_clean[pair_cols],
    figsize=(9, 9),
    diagonal='hist',
    alpha=0.35,
    color='#4C78A8'
)
plt.suptitle('Scatter Matrix of Flight Numeric Features', y=1.02)
plt.show()

##### 1. Why did you pick the specific chart?

A scatter matrix gives a compact view of several numeric pairwise relationships.

##### 2. What is/are the insight(s) found from the chart?

Distance, time, and price show structured relationships rather than random noise.

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

1. Flight class has a statistically significant impact on flight price.
2. Average flight price differs significantly across agencies.
3. Flight distance has a statistically significant positive relationship with flight price.

### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

H0: Mean price is the same for economy and premium flight types. H1: Mean price is different between economy and premium flight types.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
economy = flights_clean.loc[flights_clean['flightType'].str.contains('economic', case=False, na=False), 'price']
premium = flights_clean.loc[~flights_clean['flightType'].str.contains('economic', case=False, na=False), 'price']
t_stat, p_value = stats.ttest_ind(economy, premium, equal_var=False)
print({'t_statistic': t_stat, 'p_value': p_value})
print('Reject H0' if p_value < 0.05 else 'Fail to reject H0')


##### Which statistical test have you done to obtain P-Value?

Welch's independent two-sample t-test was used to compare mean prices across two flight-type groups.

##### Why did you choose the specific statistical test?

It is appropriate because the comparison is between two independent groups and does not assume equal variance.

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

H0: Mean flight price is the same across agencies. H1: At least one agency has a different mean flight price.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
agencies = [group['price'].values for _, group in flights_clean.groupby('agency')]
f_stat, p_value = stats.f_oneway(*agencies)
print({'f_statistic': f_stat, 'p_value': p_value})
print('Reject H0' if p_value < 0.05 else 'Fail to reject H0')


##### Which statistical test have you done to obtain P-Value?

One-way ANOVA was used to compare mean prices across more than two agency groups.

##### Why did you choose the specific statistical test?

ANOVA is appropriate for testing whether multiple independent categorical groups have different numeric means.

### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

H0: Distance and flight price have no linear relationship. H1: Distance and flight price have a significant linear relationship.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
correlation, p_value = stats.pearsonr(flights_clean['distance'], flights_clean['price'])
print({'pearson_correlation': correlation, 'p_value': p_value})
print('Reject H0' if p_value < 0.05 else 'Fail to reject H0')


##### Which statistical test have you done to obtain P-Value?

Pearson correlation test was used to evaluate the strength and significance of the distance-price relationship.

##### Why did you choose the specific statistical test?

Both variables are numeric and the business question is whether longer flights are associated with higher prices.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Handling Missing Values & Missing Value Imputation
missing_before = flights_clean.isna().sum()
print(missing_before[missing_before > 0])
# Model pipelines below use SimpleImputer: median for numeric fields and most_frequent for categorical fields.


#### What all missing value imputation techniques have you used and why did you use those techniques?

SimpleImputer is used in the pipeline: median imputation for numeric features and most-frequent imputation for categorical features. These choices are robust and deployment-friendly.

### 2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier treatments
def iqr_bounds(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in ['price', 'time', 'distance']:
    low, high = iqr_bounds(flights_clean[col])
    print(col, {'lower': low, 'upper': high, 'outliers': int(((flights_clean[col] < low) | (flights_clean[col] > high)).sum())})
# Outliers are retained because high prices may be valid premium or long-distance flights.


##### What all outlier treatment techniques have you used and why did you use those techniques?

Outliers were inspected using IQR bounds but retained because expensive flights can be valid observations caused by class, agency, route, or distance.

### 3. Categorical Encoding

In [ ]:
# Encode your categorical columns
categorical_features = ['from', 'to', 'flightType', 'agency']
numeric_features = ['time', 'distance', 'month', 'dayofweek', 'year']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
    ]
)
preprocessor


#### What all categorical encoding techniques have you used & why did you use those techniques?

One-hot encoding was used for categorical fields because route, class, and agency are nominal variables with no natural order.

### 4. Textual Data Preprocessing 
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

#### 1. Expand Contraction

In [ ]:
# Expand Contraction
print('No free-text NLP field is used for modeling, so contraction expansion is not applicable.')


#### 2. Lower Casing

In [ ]:
# Lower Casing
for col in ['from', 'to', 'flightType', 'agency']:
    flights_clean[col] = flights_clean[col].astype(str).str.strip()
flights_clean[categorical_features].head()


#### 3. Removing Punctuations

In [ ]:
# Remove Punctuations
print('City names contain state abbreviations in parentheses, so punctuation is meaningful and retained.')


#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
# Remove URLs & Remove words and digits contain digits
print('The datasets do not contain URL fields or free-form text requiring URL removal.')


#### 5. Removing Stopwords & Removing White spaces

In [ ]:
# Remove Stopwords
print('Stopword removal is not applicable because no natural-language text is modeled.')


In [ ]:
# Remove White spaces
for col in ['from', 'to', 'flightType', 'agency']:
    flights_clean[col] = flights_clean[col].astype(str).str.strip()
print('Whitespace stripped from categorical columns.')


#### 6. Rephrase Text

In [ ]:
# Rephrase Text
print('Text rephrasing is not applicable to structured travel records.')


#### 7. Tokenization

In [ ]:
# Tokenization
print('Tokenization is not required because route, class, and agency are categorical variables.')


#### 8. Text Normalization

In [ ]:
# Normalizing Text (i.e., Stemming, Lemmatization etc.)
print('Stemming and lemmatization are not applicable to this structured dataset.')


##### Which text normalization technique have you used and why?

Text normalization is not applicable because this is a structured tabular dataset, not an NLP dataset.

#### 9. Part of speech tagging

In [ ]:
# POS Taging
print('Part-of-speech tagging is not applicable because no sentence-level text is modeled.')


#### 10. Text Vectorization

In [ ]:
# Vectorizing Text
print('Categorical variables are vectorized with OneHotEncoder inside the sklearn preprocessing pipeline.')


##### Which text vectorization technique have you used and why?

OneHotEncoder is used for categorical vectorization because it preserves nominal category identity and handles unseen categories safely.

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Manipulate Features to minimize feature correlation and create new features
model_data = flights.copy()
model_data['date'] = pd.to_datetime(model_data['date'], format='%m/%d/%Y', errors='coerce')
model_data['month'] = model_data['date'].dt.month
model_data['dayofweek'] = model_data['date'].dt.dayofweek
model_data['year'] = model_data['date'].dt.year
model_data = model_data.drop(columns=['travelCode', 'userCode', 'date'])
model_data.head()


#### 2. Feature Selection

In [ ]:
# Select your features wisely to avoid overfitting
target = 'price'
features = ['from', 'to', 'flightType', 'time', 'distance', 'agency', 'month', 'dayofweek', 'year']
X = model_data[features]
y = model_data[target]
print('Selected features:', features)


##### What all feature selection methods have you used  and why?

Feature selection used business relevance and leakage prevention. travelCode and userCode were removed, while route, class, agency, time, distance, and date-derived features were retained.

##### Which all features you found important and why?

The important features are distance, time, origin, destination, flight class, agency, and calendar fields because they directly describe flight cost drivers.

### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

In [ ]:
# Transform Your data
print('Numeric scaling and categorical one-hot transformation are handled inside the ColumnTransformer pipeline.')


### 6. Data Scaling

In [ ]:
# Scaling your data
print('StandardScaler is applied to numeric features inside the modeling pipeline.')


StandardScaler is used for numeric features so linear models train on comparable numeric scales. Tree models are less sensitive, but a shared pipeline keeps preprocessing consistent.

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?

Dimensionality reduction is not necessary because the dataset is tabular, interpretable, and manageable after one-hot encoding.

In [ ]:
# DImensionality Reduction (If needed)
print('Dimensionality reduction was not applied because the encoded feature space is manageable and tree models handle sparse one-hot features well.')


##### Which dimensionality reduction technique have you used and why? (If dimensionality reduction done on dataset.)

No dimensionality reduction technique was used. Keeping encoded features supports interpretability and strong tree-model performance.

### 8. Data Splitting

In [ ]:
# Split your data to train and test. Choose Splitting ratio wisely.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)


##### What data splitting ratio have you used and why? 

An 80:20 train-test split was used. It keeps enough data for training while preserving a meaningful holdout set for validation.

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

The main target is continuous flight price, so class imbalance does not apply to the regression problem.

In [ ]:
# Handling Imbalanced Dataset (If needed)
print('The primary task is regression, so class imbalance treatment is not required for flight price prediction.')


##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)

No imbalance handling was needed for the regression target. For the gender classifier, class_weight='balanced' can be used if the classes are uneven.

## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
# ML Model - 1 Implementation
baseline_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DummyRegressor(strategy='mean'))
])
baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)

model_results = []
def record_result(name, y_true, y_pred):
    result = {
        'model': name,
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': mean_squared_error(y_true, y_pred, squared=False),
        'R2': r2_score(y_true, y_pred)
    }
    model_results.append(result)
    return result

record_result('Dummy Baseline', y_test, baseline_pred)
pd.DataFrame(model_results)


Model 1 is a DummyRegressor baseline. It predicts the average training price and establishes the minimum performance that useful models must beat.

In [ ]:
# Visualizing evaluation Metric Score chart
results_df = pd.DataFrame(model_results)
ax = results_df.set_index('model')[['MAE', 'RMSE']].plot(kind='bar', figsize=(8, 4))
ax.set_title('Baseline Error Metrics')
ax.set_ylabel('Error')
plt.xticks(rotation=0)
plt.show()


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques
print('The dummy baseline has no meaningful hyperparameters. It is used only as a reference point.')


##### Which hyperparameter optimization technique have you used and why?

No hyperparameter optimization was applied to the dummy baseline because it is only a benchmark.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

No improvement is expected from the baseline. It exists to quantify the value added by real predictive models.

### ML Model - 2

Model 2 is Linear Regression with the same preprocessing pipeline. It provides an interpretable benchmark for linear relationships among distance, class, route, agency, and price.

In [ ]:
# Visualizing evaluation Metric Score chart
linear_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_test)
record_result('Linear Regression', y_test, linear_pred)

results_df = pd.DataFrame(model_results).drop_duplicates('model', keep='last')
ax = results_df.set_index('model')[['MAE', 'RMSE']].plot(kind='bar', figsize=(8, 4))
ax.set_title('Linear Regression vs Baseline')
ax.set_ylabel('Error')
plt.xticks(rotation=20, ha='right')
plt.show()
display(results_df)


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques
cv_scores = cross_val_score(linear_model, X_train, y_train, cv=3, scoring='neg_mean_absolute_error')
print('Linear Regression CV MAE:', -cv_scores.mean())


##### Which hyperparameter optimization technique have you used and why?

Cross-validation was used for Linear Regression because it estimates generalization stability without a large hyperparameter search space.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Linear Regression should improve strongly over the dummy baseline if price has a structured relationship with the selected features.

#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

MAE is easy to explain as average price error. RMSE penalizes large pricing mistakes. R2 measures how much price variation the model explains. Lower MAE/RMSE and higher R2 improve customer quote accuracy and pricing reliability.

### ML Model - 3

In [ ]:
# ML Model - 3 Implementation
rf_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=120, min_samples_leaf=2, random_state=42, n_jobs=-1))
])
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
record_result('Random Forest', y_test, rf_pred)

results_df = pd.DataFrame(model_results).drop_duplicates('model', keep='last').sort_values('RMSE')
display(results_df)


Model 3 is Random Forest Regressor. It captures nonlinear interactions among route, class, agency, duration, and distance, making it well suited for this dataset.

In [ ]:
# Visualizing evaluation Metric Score chart
ax = results_df.set_index('model')[['MAE', 'RMSE']].plot(kind='bar', figsize=(9, 4))
ax.set_title('Regression Model Error Comparison')
ax.set_ylabel('Error')
plt.xticks(rotation=20, ha='right')
plt.show()


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 3 Implementation with hyperparameter optimization techniques
param_grid = {
    'model__n_estimators': [80, 120],
    'model__min_samples_leaf': [1, 2],
    'model__max_depth': [None, 18],
}
rf_search = GridSearchCV(
    rf_model,
    param_grid=param_grid,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)
rf_search.fit(X_train, y_train)
best_model = rf_search.best_estimator_
best_pred = best_model.predict(X_test)
record_result('Tuned Random Forest', y_test, best_pred)
print('Best parameters:', rf_search.best_params_)
display(pd.DataFrame(model_results).drop_duplicates('model', keep='last').sort_values('RMSE'))


##### Which hyperparameter optimization technique have you used and why?

GridSearchCV was used because the search space is small and the selected Random Forest parameters are important for controlling bias, variance, and runtime.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

The tuned Random Forest is expected to perform best because it captures nonlinear fare bands and route-specific effects.

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

MAE and RMSE were prioritized because pricing errors directly translate into customer-facing quote errors. R2 was also considered to measure overall explanatory power.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

The tuned Random Forest was selected as the final prediction model because it produced the strongest error metrics and can model complex route, class, agency, and distance interactions.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

The selected model is a tree ensemble. Feature importance can be inspected from the RandomForestRegressor after preprocessing; route, distance, time, class, and agency are expected to dominate because they define fare structure.

## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
# Save the File
final_model_path = MODELS_DIR / 'notebook_flight_price_model.joblib'
joblib.dump(best_model, final_model_path)
print(f'Saved model to {final_model_path}')


### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
# Load the File and predict unseen data.
loaded_model = joblib.load(final_model_path)
unseen_sample = X_test.head(3).copy()
predictions = loaded_model.predict(unseen_sample)
display(unseen_sample.assign(predicted_price=np.round(predictions, 2)))


### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

The ML workflow successfully converts raw travel records into deployable predictive assets. The flight price regression pipeline handles missing values, categorical encoding, scaling, training, evaluation, tuning, and serialization. The Random Forest model is selected because it best captures the structured pricing patterns in the data. Additional gender classification and hotel recommendation components extend the capstone into classification and recommendation use cases, while Flask, Docker, Kubernetes, Airflow, Jenkins, MLflow, and Streamlit provide the operational layer required for MLOps delivery.

The machine learning solution successfully predicts flight prices using route, agency, class, distance, time, and calendar features. A baseline model, Linear Regression, Random Forest, and tuned Random Forest were compared using MAE, RMSE, and R2. The tuned Random Forest is the final model because it captures nonlinear fare bands and route-specific effects with the best validation performance. The notebook also documents hypothesis testing, preprocessing, feature engineering, model saving, and reload-based sanity checks, making the solution ready to connect with the Flask API, Docker image, Kubernetes manifests, Airflow DAG, Jenkins pipeline, and MLflow tracking already provided in the project.

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***